# LFP dataset: Correlation analysis

Adapted from the supplied dataset-visualization workflow. This notebook uses the
existing MAT files in `../../dataset`; no duplicate dataset copy is needed.
Figures: `dataset_visualization/outputs/correlation`. Numerical source tables and
input hashes: `dataset_visualization/results/correlation`.
Run all cells in order. See the parent README for definitions and interpretation.


In [ ]:
from pathlib import Path
import sys

# Resolve from a notebook subfolder, the LFP module, or the release root.
roots = (Path.cwd(), *Path.cwd().parents)
candidates = [p for root in roots for p in
              (root, root / "dataset_visualization", root / "LFP dataset/dataset_visualization")]
PROJECT_ROOT = next(p for p in candidates if (p / "notebook_setup.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from notebook_setup import get_dataset_path, save_figure, save_table, finish_category, begin_category
CATEGORY = 'correlation'
begin_category(CATEGORY)


import scipy.io as sio 
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import matplotlib as mpl
from pathlib import Path

# =============================================================================
# Configuration and Constants
# =============================================================================
class Config:
    """Configuration parameters for the analysis."""
    def __init__(self, v_tail_upp):
        # Data paths
        self.MAT_PATH = get_dataset_path("LFP_data_processed.mat")
        
        self.V_TAIL_RANGE = (2.50, v_tail_upp)
        self.V_SEGMENT_RANGE = (v_tail_upp, v_tail_upp + 0.30)
        self.N_SEG_SUBBINS = 6
        
        # Visualization settings
        self.FIGURE_SIZE = (10, 5)
        self.FONT_FAMILY = "Arial"
        self.FONT_SIZE = 16

        self.MACARON_COLORS = [
            "#FFB3BA",
            "#FFDFBA",
            "#FFFFBA",
            "#BAFFC9",
            "#BAE1FF",
            "#E0BBE4",
            "#FFDFD3",
            "#CDE7BE",
            "#F1C0E8",
            "#B5EAD7",
            "#FFDAC1",
        ]

        self.ALPHA_VIOLIN = 0.4
        self.ALPHA_SCATTER = 0.6
        self.SCATTER_SIZE = 16
        self.JITTER_WIDTH = 0.18


# =============================================================================
# Data Loading and Processing
# =============================================================================
class BatteryDataAnalyzer:
    """Main class for battery data analysis and visualization."""
    
    def __init__(self):
        self.config = None
        self.data = None
        self.Qmax = None
        self.n_rows = None
        self.n_cells = None
        
    def load_data(self, mat_path):
        """Load and preprocess battery data once for the plotting workflow."""
        self.data = sio.loadmat(mat_path)
        
        processed_Qd_mat = self.data["processed_Qd_mat"].astype(float)
        processed_Vd_mat = self.data["processed_Vd_mat"].astype(float)
        
        self.Qmax = processed_Qd_mat[-1, :].ravel()
        self.n_rows, self.n_cells = processed_Qd_mat.shape
        
        return processed_Qd_mat, processed_Vd_mat
    
    def identify_tail_indices(self, V_matrix):
        """Identify row indices for tail voltage region."""
        V_mean = V_matrix.mean(axis=1)
        tail_mask = (V_mean >= self.config.V_TAIL_RANGE[0]) & \
                    (V_mean <= self.config.V_TAIL_RANGE[1])
        tail_indices = np.where(tail_mask)[0]
        
        print(f"Tail ({self.config.V_TAIL_RANGE[0]:.2f}-{self.config.V_TAIL_RANGE[1]:.2f} V): "
              f"{len(tail_indices)} rows")
        return tail_indices, V_mean
    
    @staticmethod
    def calculate_correlations(Q_matrix, Qmax_vector, row_indices):
        """Calculate Pearson correlations between Q rows and Qmax."""
        correlations = []
        
        for idx in row_indices:
            q_row = Q_matrix[idx, :].ravel()
            valid_mask = np.isfinite(q_row) & np.isfinite(Qmax_vector)
            
            if np.sum(valid_mask) > 1 and np.std(q_row[valid_mask]) > 0:
                r_value, _ = pearsonr(q_row[valid_mask], Qmax_vector[valid_mask])
                correlations.append(r_value)
        
        return np.array(correlations)


# =============================================================================
# Visualization: Combined Tail + Segment Sub-bins
# =============================================================================
class EnhancedNaturePlotter:
    """Creates Nature-style visualizations."""
    
    def __init__(self, config):
        self.config = config
        self._setup_style()
    
    def _setup_style(self):
        """Configure matplotlib for Nature-style plotting."""
        plt.rcParams.update({
            "font.family": self.config.FONT_FAMILY,
            "font.size": self.config.FONT_SIZE,
            "axes.linewidth": 1.2,
            "lines.linewidth": 1.5,
        })
    
    def create_tail_segment_subbins_plot(self, tail_corrs, seg_corrs_list, seg_edges):
        """Plot correlation distributions for the tail and upstream segment sub-bins."""
        fig, ax = plt.subplots(figsize=self.config.FIGURE_SIZE)
        
        data_groups = [tail_corrs] + seg_corrs_list
        
        n_groups = len(data_groups)
        positions = np.arange(1, n_groups + 1)
        
        macaron = self.config.MACARON_COLORS
        if len(macaron) < n_groups:
            raise ValueError(f"MACARON_COLORS requires at least {n_groups} colors; found {len(macaron)}.")
        colors = macaron[:n_groups]
        
        vp = ax.violinplot(
            data_groups,
            positions=positions,
            showmeans=False,
            showmedians=False,
            showextrema=False,
            widths=0.5
        )
        
        for i, body in enumerate(vp['bodies']):
            body.set_facecolor(colors[i])
            body.set_edgecolor('black')
            body.set_alpha(self.config.ALPHA_VIOLIN)
        
        boxprops = dict(linewidth=1.3, color='black')
        medianprops = dict(linewidth=2.0, color='black')
        whiskerprops = dict(linewidth=1.0, color='black')
        capprops = dict(linewidth=1.0, color='black')
        
        bp = ax.boxplot(
            data_groups,
            positions=positions,
            widths=0.18,
            vert=True,
            showfliers=False,
            patch_artist=True,
            boxprops=boxprops,
            medianprops=medianprops,
            whiskerprops=whiskerprops,
            capprops=capprops,
        )
        
        for i, box in enumerate(bp['boxes']):
            box.set_facecolor(colors[i])
            box.set_alpha(0.5)
        
        rng = np.random.default_rng(42)
        for pos, vals, col in zip(positions, data_groups, colors):
            if len(vals) == 0:
                continue
            x_jitter = pos + (rng.random(len(vals)) - 0.5) * self.config.JITTER_WIDTH
            ax.scatter(
                x_jitter, vals,
                color=col,
                s=self.config.SCATTER_SIZE,
                alpha=self.config.ALPHA_SCATTER,
                edgecolors='black',
                linewidth=0.6,
                zorder=3,
            )
        
        xticklabels = []
        xticklabels.append(f"Tail\n{self.config.V_TAIL_RANGE[0]:.2f}–{self.config.V_TAIL_RANGE[1]:.2f} V")
        for k in range(len(seg_corrs_list)):
            v_low = seg_edges[k]
            v_high = seg_edges[k+1]
            xticklabels.append(f"Frag{k+1}\n{v_low:.2f}–{v_high:.2f} V")
        
        ax.set_xticks(positions)
        ax.set_xticklabels(xticklabels, rotation=45, ha="right")
        
        ax.set_ylabel(r"Correlation coefficient")
        ax.set_ylim(-0.1, 1.2)
        
        ax.grid(False)
        
        fig.tight_layout()
        return fig, ax


# =============================================================================
# Main Execution (Automated Batch Processing)
# =============================================================================
if __name__ == "__main__":
    
    print("\nAll figures have been generated and saved.")
    
    base_config = Config(2.73)
    analyzer = BatteryDataAnalyzer()
    
    Q_matrix, V_matrix = analyzer.load_data(base_config.MAT_PATH)
    
    v_tail_upp_list = [2.73, 2.75, 2.77]
    
    for v_tail_upp in v_tail_upp_list:
        print(f"\n{'='*50}")
        print(f"Processing and plotting V_tail_upp = {v_tail_upp} V...")
        print(f"{'='*50}")
        
        config = Config(v_tail_upp=v_tail_upp)
        
        analyzer.config = config
        plotter = EnhancedNaturePlotter(config)
        
        tail_idx, V_mean = analyzer.identify_tail_indices(V_matrix)
        
        seg_low, seg_high = config.V_SEGMENT_RANGE
        seg_edges = np.linspace(seg_low, seg_high, config.N_SEG_SUBBINS + 1)
        
        seg_idx_list = []
        for k in range(config.N_SEG_SUBBINS):
            v_l = seg_edges[k]
            v_r = seg_edges[k+1]
            if k < config.N_SEG_SUBBINS - 1:
                mask = (V_mean >= v_l) & (V_mean < v_r)
            else:
                mask = (V_mean >= v_l) & (V_mean <= v_r)
            idxs = np.where(mask)[0]
            seg_idx_list.append(idxs)
            print(f"Segment {k+1}: {v_l:.2f}–{v_r:.2f} V -> {len(idxs)} rows")
        
        tail_corrs = analyzer.calculate_correlations(Q_matrix, analyzer.Qmax, tail_idx)
        seg_corrs_list = [
            analyzer.calculate_correlations(Q_matrix, analyzer.Qmax, idxs)
            for idxs in seg_idx_list
        ]
        
        print(f"\nCorrelation analysis (V_tail_upp = {v_tail_upp}):")
        print(f"  Tail region: {tail_corrs.size} valid correlations "
              f"(mean = {tail_corrs.mean():.3f} ± {tail_corrs.std():.3f})")
        for k, seg_corrs in enumerate(seg_corrs_list, start=1):
            if seg_corrs.size > 0:
                print(f"  Segment {k}: {seg_corrs.size} valid correlations "
                      f"(mean = {seg_corrs.mean():.3f} ± {seg_corrs.std():.3f})")
            else:
                print(f"  Segment {k}: 0 valid correlations")
        
        for label, rows, values in zip(['Tail']+[f'Frag{k+1}' for k in range(len(seg_idx_list))], [tail_idx]+seg_idx_list, [tail_corrs]+seg_corrs_list):
            eligible = []
            for row in rows:
                valid = np.isfinite(Q_matrix[row]) & np.isfinite(analyzer.Qmax)
                if valid.sum() > 1 and np.std(Q_matrix[row,valid]) > 0:
                    eligible.append(row)
            save_table(CATEGORY, f'{label}_correlations_Vtail_{v_tail_upp}.csv', processed_row_zero_based=eligible, mean_voltage_V=V_mean[eligible], pearson_r=values)
        fig_combined, ax_combined = plotter.create_tail_segment_subbins_plot(
            tail_corrs, seg_corrs_list, seg_edges
        )
        
        save_name = f"tail_segment_subbins_correlation_Vtail_{v_tail_upp}.png"
        save_figure(fig_combined, CATEGORY, save_name, dpi=300)
        print(f"Saved figure: {save_name}")
        
        plt.close(fig_combined)

    print("\nAll figures have been generated and saved.")

finish_category(CATEGORY)
